# QP@10 with the true pair excluded

Recomputed from the saved Version 2 test embeddings (`p3/emb`), no retraining. The last cell confirms the embeddings come from Version 2 (E3 seed 2, pair included = 0.2928, as printed in the main notebook).

In [1]:
import glob
print(glob.glob("/kaggle/input/**/p3/emb/*.npz", recursive=True)[:3])
print(glob.glob("/kaggle/input/**/pairs_70_15_15.csv", recursive=True))

['/kaggle/input/notebooks/namitabhatta/merge-project/p3/emb/E3_s0.npz', '/kaggle/input/notebooks/namitabhatta/merge-project/p3/emb/E3_s1.npz', '/kaggle/input/notebooks/namitabhatta/merge-project/p3/emb/E3_s2.npz']
['/kaggle/input/notebooks/namitabhatta/merge-project/p0/pairs_70_15_15.csv']


In [2]:
import glob, numpy as np, pandas as pd
P = glob.glob("/kaggle/input/**/p3/emb", recursive=True)[0]
pairs = pd.read_csv(glob.glob("/kaggle/input/**/pairs_70_15_15.csv", recursive=True)[0])
te = np.where(pairs.split.values == "test")[0]
y = pairs.Quadrant.map({"Q1": 0, "Q2": 1, "Q3": 2, "Q4": 3}).values[te]

def qp10_excl(ua, ul):                      # same-quadrant share of top 10, true pair excluded
    za = ua / np.linalg.norm(ua, axis=1, keepdims=True); zl = ul / np.linalg.norm(ul, axis=1, keepdims=True)
    out = []
    for M in (za @ zl.T, zl @ za.T):
        M = M.copy(); np.fill_diagonal(M, -np.inf)
        top = np.argsort(-M, 1)[:, :10]
        out.append((y[top] == y[:, None]).mean())
    return float(np.mean(out))

rows = [dict(exp=e, seed=s, qp=qp10_excl(np.load(f"{P}/{e}_s{s}.npz")["ua"][te].astype(np.float32),
                                         np.load(f"{P}/{e}_s{s}.npz")["ul"][te].astype(np.float32)))
        for e in ("E3", "E4") for s in range(5)]
R = pd.DataFrame(rows); print(R.groupby("exp").qp.agg(["mean", "std"]).round(4))
d = R[R.exp == "E4"].qp.values - R[R.exp == "E3"].qp.values
print("paired E4 - E3:", round(d.mean(), 4), "±", round(d.std(ddof=1), 4))

       mean     std
exp                
E3   0.3855  0.0709
E4   0.5948  0.0219
paired E4 - E3: 0.2092 ± 0.0759


In [3]:
Z = np.load(f"{P}/E3_s2.npz")
ua, ul = Z["ua"][te].astype(np.float32), Z["ul"][te].astype(np.float32)
za, zl = ua / np.linalg.norm(ua, axis=1, keepdims=True), ul / np.linalg.norm(ul, axis=1, keepdims=True)
qp = np.mean([(y[np.argsort(-M, 1)[:, :10]] == y[:, None]).mean() for M in (za @ zl.T, zl @ za.T)])
print("E3 seed 2 QP@10 incl. pair:", round(qp, 4), "-> Version 2 if ~0.2928, Version 1 if ~0.2265")

E3 seed 2 QP@10 incl. pair: 0.2928 -> Version 2 if ~0.2928, Version 1 if ~0.2265
